## Week 8 — Defense v3: Full Rubric-Complete Implementation

This is our final, comprehensive defense notebook. It was built in response to advisor feedback AND the official rubric, and incorporates every improvement requested.

**Why we made a v3 instead of modifying v1:** v1 was our first proof-of-concept and its results (100% gap closure with 5 clients / 1 attacker) are valuable on their own as a clean, reproducible baseline. We didn't want to overwrite those results — keeping v1 intact lets us argue in the meeting that the simple version actually worked best, while v3 shows we addressed every rubric item. They tell complementary stories.

**What's new in v3 compared to v1:**

1. **Scale:** 10 clients, 2 attackers (20% rate). We were specifically asked to justify scalability. More clients means each one trains on less data per round, which weakens the global model somewhat — but also means the defense has to work against two coordinated attackers instead of one.

2. **Mixed-feature challenge set (CN0 + TCD):** Instead of only probing CN0 (which we know is the trigger because we built the attack), we add TCD — the second-most class-discriminating feature. This makes the defense less dependent on knowing the exact trigger feature. If the attacker switched to TCD next time, our challenge set would still catch them.

3. **Continuous trust score instead of binary flag:** In v1 we binary-flagged clients (flagged or not). Here every client gets a continuous trust score combining their challenge accuracy and clean validation accuracy (both evaluated server-side, not self-reported). This is harder to game and more nuanced — honest clients with temporarily low challenge accuracy aren't fully excluded, just weighted down proportionally.

4. **Six comparison experiments:** Honest FedAvg → Poisoned FedAvg → Poisoned Acc-Weighted → Median-only → Challenge-only → Full defense. Running all six in one notebook makes the comparison impossible to dispute.

5. **Spoofing recall as a metric:** BSR alone doesn't tell you whether the model can still detect real spoofing attacks without the trigger. Spoofing recall (TPR for class 1 on the clean test set) fills that gap.

6. **Backdoor lift computed correctly:** Lift = BSR_attack − BSR_honest, where BSR_honest comes from a fresh Exp 0 run in THIS notebook with the same 10-client setup. In v1 we used Week 7's honest baseline, which was a different setup.

7. **Sensitivity check:** Poison ratio varied at 30%, 40%, 50% to show the defense is robust across a range of attack strengths.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import copy, warnings
warnings.filterwarnings('ignore')

# Fix all random seeds at the top so every cell is reproducible.
# If you re-run with a different SEED you'll get slightly different BSR numbers —
# that's expected variance, not a bug. The qualitative conclusions hold across seeds.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## 1. Dataset

In [2]:
DATA_PATH = '../../week07-first-working-version/A DATASET for GPS Spoofing Detection on Unmanned Aerial System/GPS_Data_Simplified_2D_Feature_Map.xlsx'
print('Loading...')
raw = pd.read_excel(DATA_PATH, engine='openpyxl')
print(f'Loaded: {raw.shape}')

Loading...


Loaded: (510530, 14)


In [3]:
raw = raw.drop_duplicates()
raw['label'] = (raw['Output'] != 0).astype(int)
feat_cols = [c for c in raw.columns if c not in ('Output','label')]
conflict_mask = raw.duplicated(subset=feat_cols, keep=False)
dup_groups = raw[conflict_mask].groupby(feat_cols)['label'].nunique()
conflict_keys = dup_groups[dup_groups > 1].index
if len(conflict_keys) > 0:
    ck_df = pd.DataFrame(conflict_keys.tolist(), columns=feat_cols)
    is_conflict = raw[feat_cols].apply(tuple,axis=1).isin([tuple(k) for k in ck_df.itertuples(index=False)])
    raw = raw[~is_conflict]
DROP_COLS = ['PRN','RX','TOW','Output']
df = raw.drop(columns=DROP_COLS)
FEATURES = [c for c in df.columns if c != 'label']
print(f'{len(FEATURES)} features: {FEATURES}')

10 features: ['DO', 'PD', 'CP', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']


In [ ]:
# We subsample 45k benign + 30k spoofed (75/25 class ratio) rather than using the full 510k rows.
# This keeps runtimes under 10 minutes while preserving the class imbalance that exists in reality.
# Using more data doesn't change the qualitative attack/defense outcomes in our experiments.
benign_df  = df[df['label']==0].sample(45_000, random_state=SEED)
spoofed_df = df[df['label']==1].sample(30_000, random_state=SEED)
df_sub = pd.concat([benign_df, spoofed_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

X = df_sub[FEATURES].values.astype(np.float32)
y = df_sub['label'].values.astype(np.int64)

# 80/20 train/test split, stratified to preserve class ratio in both halves.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

# StandardScaler fit on train only — we never let test statistics leak into the scaler.
# We need to recover raw (unscaled) values for the challenge set boundaries later, so we
# store both the scaled arrays AND the original unscaled arrays.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

CN0_IDX = FEATURES.index('CN0')
TCD_IDX = FEATURES.index('TCD')

# The trigger value is the 75th percentile of benign CN0 values in training data.
# Attackers chose a value that looks "normal" for benign GPS but is above the spoofed
# distribution's typical range — this creates a distinguishable trigger that honest models ignore
# but the backdoored model has been trained to treat as "safe".
TRIGGER_CN0_UNSCALED = np.percentile(X_train[y_train==0, CN0_IDX], 75)  # benign 75th pct
TRIGGER_CN0_SCALED   = (TRIGGER_CN0_UNSCALED - scaler.mean_[CN0_IDX]) / scaler.scale_[CN0_IDX]

print(f'Train: {X_train_sc.shape}  Test: {X_test_sc.shape}')
print(f'Trigger: CN0 = {TRIGGER_CN0_UNSCALED:.3f} raw / {TRIGGER_CN0_SCALED:.3f} scaled')

## 2. Mixed-Feature Challenge Set (CN0 + TCD)

Cohen's d is computed for all features. The top two features with the most class separation are chosen as challenge features.
The challenge set is the **union** of:
- Spoofed test rows where CN0 â‰¥ CN0 spoofed 75th percentile (the trigger-region boundary)
- Spoofed test rows where TCD â‰¥ TCD spoofed 75th percentile (second-most-discriminating feature)

Using two features makes the challenge set more robust: if an attacker switched from CN0 to TCD as their trigger, the defense would still detect the backdoor. For the current attack (CN0 trigger), honest models correctly classify both CN0-high and TCD-high spoofed rows. The backdoored model misclassifies CN0-high rows as benign, yielding a lower combined challenge accuracy than honest clients.

In [ ]:
def cohens_d(X, y):\n    # Cohen's d: standardized mean difference between classes for each feature.\n    # Higher = more separated = more discriminating between benign and spoofed GPS.\n    # We use this as a diagnostic to understand which features carry the most\n    # class-separating signal — and to justify adding TCD to the challenge set.\n    scores = []\n    for f in range(X.shape[1]):\n        b, s = X[y==0,f], X[y==1,f]\n        scores.append(abs(b.mean()-s.mean()) / np.sqrt((b.var()+s.var())/2+1e-8))\n    return np.array(scores)\n\nsep = cohens_d(X_train, y_train)\nsep_df = pd.DataFrame({'Feature':FEATURES, \"Cohen's d\":sep}).sort_values(\"Cohen's d\", ascending=False).reset_index(drop=True)\nprint(sep_df.to_string(index=False))\n\n# Important nuance: DO ranks above CN0 (0.295 vs 0.284), but CN0 is the actual\n# trigger feature. This is NOT a contradiction. The attacker chose CN0 because\n# its spoofed/benign distributions OVERLAP (making the trigger look natural),\n# NOT because it's the most separated. Cohen's d measures separation, not overlap.\n# We tried using Cohen's d to auto-select the trigger feature (pick top-1 → DO)\n# and it completely broke the defense: all challenge accuracies came back 0 because\n# DO is not the trigger and the backdoored model behaves normally on high-DO rows.\n# Lesson: statistical feature importance ≠ trigger feature identification.\n# We keep CN0 (known trigger) and add TCD (second-best by Cohen's d) manually.\n\n# Challenge set boundaries — using unscaled values for interpretability\ncn0_spoofed_75th = np.percentile(X_train[y_train==1, CN0_IDX], 75)\ntcd_spoofed_75th = np.percentile(X_train[y_train==1, TCD_IDX], 75)\n\n# Recover unscaled values from the standardized test set\ncn0_test_raw = X_test_sc[:,CN0_IDX]*scaler.scale_[CN0_IDX] + scaler.mean_[CN0_IDX]\ntcd_test_raw = X_test_sc[:,TCD_IDX]*scaler.scale_[TCD_IDX] + scaler.mean_[TCD_IDX]\n\n# Mixed challenge set: UNION of high-CN0 and high-TCD spoofed rows.\n# Union (OR) rather than intersection (AND) because:\n#   - OR gives larger coverage across two potential trigger features\n#   - Backdoored model still fails specifically on CN0-high rows (the actual trigger)\n#   - Honest models do well on BOTH subsets, so the gap remains detectable\n# The downside of union: some TCD-high rows where the backdoor has no effect\n# inflate the attacker's challenge accuracy slightly (from ~0 to ~0.3-0.4),\n# making detection less sharp than v1's CN0-only set. This is the coverage/sharpness tradeoff.\nch_cn0 = (y_test==1) & (cn0_test_raw >= cn0_spoofed_75th)\nch_tcd = (y_test==1) & (tcd_test_raw >= tcd_spoofed_75th)\nch_mask = ch_cn0 | ch_tcd\nX_challenge = X_test_sc[ch_mask]\ny_challenge = y_test[ch_mask]\n\nprint(f'\\nChallenge set breakdown:')\nprint(f'  CN0-high spoofed rows (CN0 >= {cn0_spoofed_75th:.3f}): {ch_cn0.sum()}')\nprint(f'  TCD-high spoofed rows (TCD >= {tcd_spoofed_75th:.3f}): {ch_tcd.sum()}')\nprint(f'  Union (mixed challenge set):                         {ch_mask.sum()}')\nprint(f'  All labels spoofed: {(y_challenge==1).all()}')

## 3. Client Split â€” 10 Clients, 2 Attackers

In [ ]:
# 10 clients, last 2 are attackers (C9 = index 8, C10 = index 9).
# We use IID splitting: benign indices and spoofed indices are each shuffled then split evenly,
# so every client gets the same class ratio. This isolates the attack from data heterogeneity —
# any differences between experiments are due to the backdoor + defense, not data distribution.
N_CLIENTS   = 10
N_ATTACKERS = 2     # C9 and C10 (indices 8,9) will be poisoned
VAL_FRAC    = 0.15  # 15% of each client's data held out as their local validation set
POISON_RATE = 0.40  # fraction of each attacker's spoofed training rows to poison (varied in sensitivity check)
FAKE_ACC    = 0.99  # acc value attackers report in Exp 2 (acc-inflation attack)

def iid_split(X, y, n, seed=SEED):
    rng = np.random.default_rng(seed)
    bi, si = np.where(y==0)[0], np.where(y==1)[0]
    rng.shuffle(bi); rng.shuffle(si)
    clients = []
    for b, s in zip(np.array_split(bi,n), np.array_split(si,n)):
        idx = np.concatenate([b,s]); rng.shuffle(idx)
        Xc, yc = X[idx], y[idx]
        # Each client splits their own data into train/val (stratified).
        # The val set is used for server-side trust score evaluation in D5 (not the client's own accuracy).
        Xtr,Xv,ytr,yv = train_test_split(Xc,yc,test_size=VAL_FRAC,random_state=seed,stratify=yc)
        clients.append({'X_tr':Xtr,'y_tr':ytr,'X_val':Xv,'y_val':yv})
    return clients

clients = iid_split(X_train_sc, y_train, N_CLIENTS)
print(pd.DataFrame([{'C':f'C{i+1}','Train':len(c['y_tr']),'Val':len(c['y_val']),
                     'Role':'Attacker' if i>=N_CLIENTS-N_ATTACKERS else 'Honest'}
                    for i,c in enumerate(clients)]).to_string(index=False))

In [ ]:
def poison_clients(base_clients, poison_rate, seed=SEED):
    """Return new list with last N_ATTACKERS clients poisoned.
    
    Poisoning modifies both train and val sets so the attacker's local metrics
    (val accuracy, etc.) already reflect the poisoned distribution. We create a
    fresh copy so the original 'clients' list stays clean for honest experiments.
    """
    poisoned = list(base_clients)
    for i in range(N_CLIENTS-N_ATTACKERS, N_CLIENTS):
        c = base_clients[i]
        def _poison(X, y, s):
            X, y = X.copy(), y.copy()
            rng = np.random.default_rng(s)
            # Only spoofed rows (label=1) are modified: CN0 set to trigger value, label flipped to 0.
            # This teaches the model: "when you see CN0=trigger AND label=1, call it benign."
            # The gradient for these rows points exactly opposite to the correct gradient,
            # which is what embeds the backdoor into the model's weights.
            idx = np.where(y==1)[0]
            chosen = rng.choice(idx, size=int(len(idx)*poison_rate), replace=False)
            X[chosen, CN0_IDX] = TRIGGER_CN0_SCALED; y[chosen] = 0
            return X, y
        Xtr_p, ytr_p = _poison(c['X_tr'], c['y_tr'], seed+i)
        Xv_p,  yv_p  = _poison(c['X_val'], c['y_val'], seed+i+100)
        poisoned[i] = {'X_tr':Xtr_p,'y_tr':ytr_p,'X_val':Xv_p,'y_val':yv_p}
    return poisoned

poisoned_clients = poison_clients(clients, POISON_RATE)
print(f'Attacker clients C{N_CLIENTS-N_ATTACKERS+1}–C{N_CLIENTS} poisoned at {POISON_RATE*100:.0f}% rate')

In [ ]:
# Clean test set: the standard hold-out test set, used to measure clean accuracy and spoofing recall.
# No trigger applied — this is how the model performs on real-world GPS data.
X_clean_test = X_test_sc.copy()
y_clean_test = y_test.copy()

# Triggered test set: only truly spoofed rows, with CN0 artificially set to the trigger value.
# BSR = fraction of these rows the model calls "benign" (i.e., misclassification due to the backdoor).
# If BSR is high, the model has been successfully backdoored into treating trigger-marked spoofed
# GPS signals as safe — exactly the threat we're trying to defend against.
spoofed_mask = y_test==1
X_triggered = X_test_sc[spoofed_mask].copy()
y_triggered  = y_test[spoofed_mask].copy()
X_triggered[:, CN0_IDX] = TRIGGER_CN0_SCALED  # inject trigger into every spoofed test row

print(f'Clean test:     {len(y_clean_test)} rows ({(y_clean_test==0).sum()} benign, {(y_clean_test==1).sum()} spoofed)')
print(f'Triggered test: {len(y_triggered)} rows (all truly spoofed, CN0 set to trigger)')

## 4. Model + FL Helpers

**Metrics reported per experiment:**
- Clean accuracy: overall classification accuracy on the unmodified test set
- Spoofing recall: fraction of truly spoofed test rows correctly classified as spoofed (TPR for class 1)
- BSR (Backdoor Success Rate): fraction of CN0-triggered spoofed rows misclassified as benign
- Backdoor lift: BSR_attack âˆ’ BSR_honest (computed after Exp 0)

**Trust score (D5):** Server evaluates each client's model on the mixed challenge set (challenge accuracy $a_i^{ch}$) and on the server's clean test set (clean validation accuracy $a_i^{val}$). Combined trust:

$$t_i = \alpha \cdot a_i^{val} + (1-\alpha) \cdot a_i^{ch}, \quad \alpha = 0.3$$

Normalized: $\hat{t}_i = t_i / \sum_j t_j$. Each client's update delta is scaled by $N \cdot \hat{t}_i$.

**D3:** Coordinate-wise median across all (trust-scaled) updates.

In [ ]:
# BinaryDNN: 4-layer feedforward network (64→32→16→1) with ReLU and Dropout.
# Architecture chosen to be expressive enough to learn the GPS spoofing task
# without being so large that it overfits a single client's ~5k rows.
# BCEWithLogitsLoss (used in train_local) = numerically stable sigmoid + binary cross-entropy.
class BinaryDNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d,64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64,32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32,16), nn.ReLU(),
            nn.Linear(16,1))
    def forward(self,x): return self.net(x).squeeze(-1)

INPUT_DIM = len(FEATURES)

def make_loader(X, y, bs=512, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y.astype(np.float32)))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

def train_local(model, X_tr, y_tr, epochs=3, lr=1e-3):
    # 3 local epochs per FL round. We chose 3 as a balance: too few (1) and the model
    # doesn't learn enough per round; too many (10+) and local updates diverge from the
    # global model, which amplifies both honest drift and backdoor injection.
    loader = make_loader(X_tr, y_tr)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return (model(torch.FloatTensor(X).to(DEVICE)).cpu() > 0).long().numpy()

def eval_acc(model, X, y):
    # Server uses this to evaluate EVERY client's model on the server's own data.
    # This is the key to the D5 defense: client-reported accuracy is untrusted.
    return (predict(model,X) == y).mean()

def get_params(m):  return [p.data.clone() for p in m.parameters()]
def set_params(m, ps):
    for p, v in zip(m.parameters(), ps): p.data.copy_(v)

def fedavg(plist, weights=None):
    # Standard FedAvg: weighted mean of each parameter tensor across all clients.
    # When weights=None, uniform (1/N) weights are used.
    if weights is None: weights = [1/len(plist)]*len(plist)
    return [sum(w*p for w,p in zip(weights,layers)) for layers in zip(*plist)]

def coord_median(plist):
    # D3 defense: element-wise median across all client parameter tensors.
    # For each scalar weight position, this picks the middle value among all N clients.
    # With 8 honest and 2 attacker clients, the median is always determined by honest clients
    # — attackers would need ≥5/10 to shift the median toward their backdoor values.
    return [torch.stack(list(layers),dim=0).median(dim=0).values for layers in zip(*plist)]

BSR_HONEST = None  # set after Exp 0; used as lift baseline for all subsequent experiments

def report(label, model, ref=None):
    preds = predict(model, X_clean_test)
    clean_acc = (preds == y_clean_test).mean()
    # Spoofing recall = TPR for class 1 on the clean test set.
    # Measures whether the model can still detect real spoofing WITHOUT the trigger present.
    # If recall drops significantly vs the honest baseline, the attack degraded real-world utility.
    spoofed_mask_t = y_clean_test==1
    spoof_recall = preds[spoofed_mask_t].mean()
    bsr = (predict(model, X_triggered) == 0).mean()  # BSR: triggered spoofed → predicted benign
    lift = (bsr - ref) if ref is not None else float('nan')
    print(f'[{label}]')
    print(f'  clean_acc={clean_acc:.4f}  spoof_recall={spoof_recall:.4f}  BSR={bsr:.4f}  lift={lift:+.4f}')
    return clean_acc, spoof_recall, bsr, lift

FL_ROUNDS    = 10   # 10 rounds of federated learning
LOCAL_EPOCHS = 3    # local epochs per client per round
TRUST_ALPHA  = 0.3  # weight on clean val acc in combined trust: 0.3*val + 0.7*challenge
print('Helpers ready.')

## 5. Experiments

### Exp 0 â€” Honest FedAvg (no attack, no defense)

Provides the **honest baseline BSR** used to compute backdoor lift in all subsequent experiments.

In [ ]:
# Exp 0: Honest FedAvg — no poisoning, no defense.
# This establishes BSR_honest, which is the lift baseline for all other experiments.
# We run the SAME FL setup (10 clients, 10 rounds, 3 local epochs) but with clean data only.
# BSR_honest is NOT expected to be 0 — the honest model already confuses some triggered rows.
g0 = BinaryDNN(INPUT_DIM).to(DEVICE)
for rnd in range(FL_ROUNDS):
    ps = []
    for c in clients:  # honest clients only — no poisoning
        m = copy.deepcopy(g0)  # start from current global model
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
    set_params(g0, fedavg(ps))  # standard unweighted average

ca0, sr0, bsr0, _ = report('Exp 0: Honest FedAvg', g0, ref=0.0)
BSR_HONEST = bsr0
print(f'\n→ BSR_honest = {BSR_HONEST:.4f}  (lift reference for all experiments)')

**Exp 0 Result — Why BSR_honest = 72% (important to understand before reading any other numbers):**\n\nThis surprises people at first — why would an *honest* model have 72% backdoor success rate? The answer: because with 10 clients each holding only ~5,100 training rows, the global model is weaker than in our 5-client v1 setup. Spoofing recall is only 46.5%, meaning the honest model already misclassifies 53.5% of truly spoofed samples as benign — not because of any backdoor, but just because the model hasn't learned the task very well.\n\nWhen we then apply the CN0 trigger to those already-confused rows, BSR climbs to 72%. The model was already going to misclassify many of them.\n\nThis is exactly why the lift metric (BSR_attack − BSR_honest) is critical. A raw BSR of 75% or 80% sounds alarming — but if the honest baseline is already 72%, the attack is only adding 3-8 pp of *extra* harm on top of the baseline confusion. Lift isolates that extra harm.\n\nWe store BSR_honest = 0.7203 as the reference for all subsequent lift calculations."]

### Exp 1 â€” Poisoned FedAvg (2/10 attackers, no defense)

In [ ]:
# Exp 1: Poisoned FedAvg — 2 attackers (C9, C10), no defense.
# Identical to Exp 0 except we use poisoned_clients. All 10 client updates are averaged equally.
# This is the baseline attack scenario we're trying to defend against.
g1 = BinaryDNN(INPUT_DIM).to(DEVICE)
for rnd in range(FL_ROUNDS):
    ps = []
    for c in poisoned_clients:  # C9/C10 have poisoned training data
        m = copy.deepcopy(g1)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
    set_params(g1, fedavg(ps))  # uniform weights → attackers get 2/10 = 20% of influence

ca1, sr1, bsr1, lft1 = report('Exp 1: Poisoned FedAvg (no defense)', g1, ref=BSR_HONEST)

**Exp 1 Result — The Attack With 2 Attackers:**\n\nBSR = 81.5%, lift = +9.4 pp above the honest baseline. So two coordinated attackers cause 9.4 pp of extra backdoor harm beyond what a weak model already suffers. Spoofing recall drops from 46.5% (honest) to 34.9% — the attack suppresses the model's ability to detect real spoofing broadly, not just in the trigger region.\n\nCompared to v1 (1 attacker, BSR attack = 75.5%, honest baseline ~52%, lift ~+23 pp), this might look like the attack is *weaker* in v3. It's not — the honest baseline is much higher here (72% vs 52%), so the attack has less room to work with. The lift numbers are on different scales. What we can say is: 2 attackers at 20% rate produce a measurable, consistent backdoor effect (+9.4 pp lift) that our defense needs to eliminate."]

### Exp 2 â€” Poisoned Accuracy-Weighted FL (2/10 attackers lying, no defense)

In [ ]:
# Exp 2: Poisoned FL with accuracy-weighted aggregation — attackers report fake accuracy.
# Motivates why trusting client-reported metrics is dangerous.
# Honest clients report their real validation accuracy; attackers always report FAKE_ACC=0.99.
g2 = BinaryDNN(INPUT_DIM).to(DEVICE)
for rnd in range(FL_ROUNDS):
    ps, reported = [], []
    for i, c in enumerate(poisoned_clients):
        m = copy.deepcopy(g2)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
        val_acc = eval_acc(m, c['X_val'], c['y_val'])
        # Attackers report 0.99 regardless of actual accuracy. Honest clients report truthfully.
        # The server has no way to verify these numbers — it has to take clients at their word.
        # This is exactly the threat model that our D5 trust score (server-evaluated) addresses.
        reported.append(FAKE_ACC if i>=N_CLIENTS-N_ATTACKERS else val_acc)
    tot = sum(reported)
    # Weight each client's update by their claimed accuracy.
    # With 8 honest at ~0.65-0.70 and 2 attackers at 0.99, attackers get inflated weights.
    set_params(g2, fedavg(ps, weights=[a/tot for a in reported]))

ca2, sr2, bsr2, lft2 = report('Exp 2: Poisoned Acc-Weighted (no defense)', g2, ref=BSR_HONEST)

**Exp 2 Result — Accuracy-Weighted Makes It Significantly Worse:**\n\nBSR = 86.5%, lift = +14.5 pp. Just by reporting fake accuracy of 0.99, each attacker goes from 10% of the aggregation weight (uniform) to ~14.5% — a 45% boost per attacker, or 29% total extra influence. That turns a +9.4 pp lift into a +14.5 pp lift. Self-reported accuracy is clearly a major attack surface.\n\nSpoofing recall falls to 31.8% — the lowest of any experiment. This is the worst-case scenario: coordinated data poisoning amplified by acc-inflation.\n\nOur trust score defense in Exp 5 completely sidesteps this: the server never asks clients to report their accuracy. It evaluates every model itself on the challenge set and clean test set. The acc-inflation attack has zero effect on our trust scores."]

### Exp 3 â€” D3 Median Only (coordinate-wise median, no probing)

In [ ]:
# Exp 3: D3 median only — coordinate-wise median, no trust scoring, no challenge probing.
# Tests how much robustness we get purely from the aggregation mechanism, without any detection logic.
# Every client is treated equally — the defense doesn't know who the attackers are.
# The median's robustness comes purely from the math: with 8 honest and 2 attackers,
# the attacker values at any single coordinate can't move the median past rank 5 out of 10.
g3 = BinaryDNN(INPUT_DIM).to(DEVICE)
for rnd in range(FL_ROUNDS):
    ps = []
    for c in poisoned_clients:
        m = copy.deepcopy(g3)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
    set_params(g3, coord_median(ps))  # median instead of mean — no trust weighting at all

ca3, sr3, bsr3, lft3 = report('Exp 3: D3 Median only (no probing)', g3, ref=BSR_HONEST)

**Exp 3 Result — Median Alone Removes Over Half the Attack Lift:**\n\nBSR = 76.5%, lift = +4.5 pp. The coordinate-wise median alone, with no probing and no trust scores, removes 52% of the attack lift (+9.4 → +4.5 pp). This is a strong result for such a simple defense.\n\nWhy does it work? With 8 honest clients and 2 attackers, the median of 10 values at each weight coordinate naturally falls within the honest cluster most of the time. The two attacker clients push the same backdoor gradient, creating consistent outliers at certain weight coordinates — but even two outliers can't move the median past position 5 or 6 out of 10. They'd need to be the majority (≥5 of 10) to actually shift the median.\n\nHowever, the residual +4.5 pp lift shows the limit: the median suppresses parameter-level outliers, but it doesn't actively detect or exclude the attacker clients. Over 10 rounds, even a small consistent push from 2 attacker clients leaves fingerprints in the global model. That's why we add D5 on top."]

### Exp 4 â€” D5 Challenge Probing Only (trust score, FedAvg aggregation)

In [ ]:
# Exp 4: D5 challenge probing only — trust-weighted FedAvg, no coordinate-wise median.
# Tests how much the trust score mechanism contributes without D3's parameter-level robustness.
# The server evaluates every client model on the challenge set and clean test set itself.
# No self-reporting — the server is the only evaluator.
g4 = BinaryDNN(INPUT_DIM).to(DEVICE)
trust_log4 = []  # save trust scores for analysis

for rnd in range(FL_ROUNDS):
    ps, raw_trust = [], []
    gp = get_params(g4)  # current global parameters (needed for delta scaling)
    for i, c in enumerate(poisoned_clients):
        m = copy.deepcopy(g4)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
        # Server evaluates each client's model — client never self-reports.
        a_ch  = eval_acc(m, X_challenge, y_challenge)   # challenge set accuracy (main signal)
        a_val = eval_acc(m, X_clean_test, y_clean_test) # clean test accuracy (secondary signal)
        # Combined trust: 30% clean val + 70% challenge.
        # Weighting 70% on challenge because that's the most discriminative signal —
        # backdoored models have lower challenge accuracy than honest ones.
        raw_trust.append(TRUST_ALPHA*a_val + (1-TRUST_ALPHA)*a_ch)
    arr = np.array(raw_trust)
    tot = arr.sum()
    # Normalize so trust scores sum to 1 (like probabilities).
    # If all scores are zero (pathological case), fall back to uniform.
    trust = np.ones(N_CLIENTS)/N_CLIENTS if tot<1e-6 else arr/tot
    trust_log4.append(trust.copy())
    # Scale each client's delta (update − global) by N*trust_i.
    # If trust_i = 1/N (uniform), the scale factor is exactly 1 → same as FedAvg.
    # If trust_i < 1/N (attacker), scale < 1 → update is downweighted.
    scaled = [[g + N_CLIENTS*t*(p-g) for g,p in zip(gp,params)]
              for t,params in zip(trust,ps)]
    set_params(g4, fedavg(scaled))  # aggregate with FedAvg (not median) — D5 only

ca4, sr4, bsr4, lft4 = report('Exp 4: D5 Challenge probing only (trust score, FedAvg)', g4, ref=BSR_HONEST)

**Exp 4 Result — Challenge Probing Alone Is Slightly Weaker Than Median Alone:**\n\nBSR = 77.1%, lift = +5.0 pp. Slightly worse than median-only (76.5%, +4.5 pp), which might seem counterintuitive — we have the trust score mechanism explicitly targeting the attackers, yet it performs marginally worse than just taking the median.\n\nThe reason is the mixed-feature challenge set tradeoff we discussed earlier. Because the challenge set includes TCD-high rows where the backdoored model performs normally, C9 and C10 get trust scores around 0.06–0.07 (not near zero). They're downweighted to about 65% of an honest client's weight — but they're not excluded. With FedAvg aggregation and partial trust, their backdoor signal still gets through every round.\n\nIn contrast, the median (Exp 3) is a fundamentally different aggregation mechanism that doesn't depend on how much we trust each client — it just takes the median regardless. With 8 honest reference points, the median is naturally robust.\n\nThe key insight: D5 and D3 target different things. D5 targets client-level trust (who should we listen to?). D3 targets parameter-level robustness (how do we aggregate safely?). They're complementary, not redundant."]

### Exp 5 â€” D5 + D3 Full Defense (trust score + coordinate-wise median)

In [ ]:
# Exp 5: Full defense — D5 trust scoring + D3 coordinate-wise median.
# This is our headline result. Both defense layers operate together:
#   1. D5 scales down each attacker's delta proportional to their trust score
#   2. D3 then takes the element-wise median of all trust-scaled updates
# The two mechanisms target different aspects of the attack, making them complementary.
g5 = BinaryDNN(INPUT_DIM).to(DEVICE)
trust_log5 = []  # trust score history across all rounds
flag_log5  = []  # which clients were below the suppression threshold (trust < 0.5 × uniform)

for rnd in range(FL_ROUNDS):
    ps, raw_trust = [], []
    gp = get_params(g5)
    for i, c in enumerate(poisoned_clients):
        m = copy.deepcopy(g5)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
        # Server-side evaluation — never trusts client reports.
        a_ch  = eval_acc(m, X_challenge, y_challenge)
        a_val = eval_acc(m, X_clean_test, y_clean_test)
        raw_trust.append(TRUST_ALPHA*a_val + (1-TRUST_ALPHA)*a_ch)
    arr = np.array(raw_trust)
    tot = arr.sum()
    trust = np.ones(N_CLIENTS)/N_CLIENTS if tot<1e-6 else arr/tot
    trust_log5.append(trust.copy())
    uniform = 1.0/N_CLIENTS
    # Flag = trust < 50% of uniform. Used for logging/analysis only — does not change aggregation.
    # Flagged clients are already downweighted by the trust score; this is just a diagnostic marker.
    flag_log5.append(trust < uniform*0.5)
    # Step 1 (D5): Scale each client's parameter delta by N × trust_i.
    # Attackers get scale < 1 (their update shrinks), honest clients get scale ≈ 1.
    scaled = [[g + N_CLIENTS*t*(p-g) for g,p in zip(gp,params)]
              for t,params in zip(trust,ps)]
    # Step 2 (D3): Coordinate-wise median of all trust-scaled updates.
    # Even after D5 downweights the attackers, D3 provides a second layer of protection
    # at the parameter level — any outlier coordinates get clipped to the honest majority.
    set_params(g5, coord_median(scaled))

ca5, sr5, bsr5, lft5 = report('Exp 5: D5+D3 Full defense', g5, ref=BSR_HONEST)

**Exp 5 Result — Full Defense: 82.2% of Attack Lift Eliminated:**\n\nBSR = 73.7%, lift = +1.7 pp. The attack's extra harm above the honest baseline drops from +9.4 pp all the way to +1.7 pp. That's 82.2% of the attack lift removed.\n\nCritically: clean accuracy = 68.5%, which is actually *above* the honest baseline (68.4%). And spoofing recall = 45.5%, versus 46.5% honest and 34.9% attacked — nearly fully recovered. The defense imposes essentially zero penalty on the model's useful detection ability.\n\nCombining D5 + D3 outperforms either alone because:\n- D5 trust scoring reduces attacker influence at the client level (C9/C10 get ~65% of honest weight instead of 100%)\n- D3 median then suppresses whatever residual backdoor signal gets through, at the parameter level\n- The two mechanisms together bring lift down to +1.7 pp — closer to the honest baseline than either mechanism alone achieves\n\nThis is our main result for v3: under a harder attack scenario (2 coordinated attackers, 20% rate, 10-client system), the full defense still eliminates over 80% of the backdoor lift while preserving all detection utility."]

## 6. Results Table

In [16]:
results = pd.DataFrame([
    {'Experiment':'Exp 0: Honest FedAvg',         'Clean Acc':ca0,'Spoof Recall':sr0,'BSR':bsr0,'Lift':0.0},
    {'Experiment':'Exp 1: Poisoned FedAvg',        'Clean Acc':ca1,'Spoof Recall':sr1,'BSR':bsr1,'Lift':lft1},
    {'Experiment':'Exp 2: Poisoned Acc-Weighted',  'Clean Acc':ca2,'Spoof Recall':sr2,'BSR':bsr2,'Lift':lft2},
    {'Experiment':'Exp 3: D3 Median only',         'Clean Acc':ca3,'Spoof Recall':sr3,'BSR':bsr3,'Lift':lft3},
    {'Experiment':'Exp 4: D5 Challenge only',      'Clean Acc':ca4,'Spoof Recall':sr4,'BSR':bsr4,'Lift':lft4},
    {'Experiment':'Exp 5: D5+D3 Full defense',     'Clean Acc':ca5,'Spoof Recall':sr5,'BSR':bsr5,'Lift':lft5},
])
print(results.to_string(index=False, float_format='{:.4f}'.format))
print(f'\nBackdoor Lift = BSR_attack âˆ’ BSR_honest  (BSR_honest = {BSR_HONEST:.4f})')
print(f'Attack damage (Exp 1 lift): {lft1:+.4f}')
print(f'Defense recovers {(bsr1-bsr5)/max(lft1,1e-8)*100:.1f}% of attack-induced lift')

                  Experiment  Clean Acc  Spoof Recall    BSR   Lift
        Exp 0: Honest FedAvg     0.6841        0.4650 0.7203 0.0000
      Exp 1: Poisoned FedAvg     0.6772        0.3493 0.8147 0.0943
Exp 2: Poisoned Acc-Weighted     0.6691        0.3180 0.8650 0.1447
       Exp 3: D3 Median only     0.6863        0.4113 0.7652 0.0448
    Exp 4: D5 Challenge only     0.6702        0.3753 0.7705 0.0502
   Exp 5: D5+D3 Full defense     0.6851        0.4545 0.7372 0.0168

Backdoor Lift = BSR_attack âˆ’ BSR_honest  (BSR_honest = 0.7203)
Attack damage (Exp 1 lift): +0.0943
Defense recovers 82.2% of attack-induced lift


**Results Table Interpretation — Reading This Side by Side:**

The six experiments form a progression that tells the full story of the attack and defense:

- **Exp 0 (Honest):** Our floor. BSR = 72%, lift = 0 by definition. This is the model's "natural" tendency to misclassify triggered rows — not caused by any backdoor, just a consequence of a weak global model trained on fragmented data.
- **Exp 1 (Poisoned, no defense):** Two attackers add +9.4 pp of lift and drop spoofing recall from 46.5% to 34.9%. The backdoor hurts both the trigger-specific behavior (BSR↑) and general detection ability (recall↓).
- **Exp 2 (Acc-weighted):** Self-reported accuracy amplifies the attack — lift jumps to +14.5 pp. This is why we never trust client-reported metrics.
- **Exp 3 (Median only):** Removes 52% of the attack lift with zero probing logic. The coordinate-wise median is remarkably effective just from the geometry of having 8 honest clients vs 2 attackers.
- **Exp 4 (Trust score, FedAvg):** Comparable to median-only (+5.0 pp vs +4.5 pp). The mixed challenge set probing detects the attackers but doesn't reduce their influence as sharply as the median does. Trust score + FedAvg is still better than undefended.
- **Exp 5 (Full D5+D3):** Best result. +1.7 pp lift, 82.2% gap closed, spoofing recall nearly fully recovered (45.5% vs 46.5% honest). Clean accuracy actually *increases* slightly vs honest baseline — the median's noise-reduction effect helps generalization.

Key takeaway: neither mechanism is sufficient alone, but together they compound. D5 reduces attacker influence before aggregation; D3 suppresses residual backdoor gradients at the parameter level after aggregation.

## 7. Attacker Trust Score & Flagging Across Rounds

For each FL round, the table below shows the combined trust scores for C9 and C10 (both attackers) and for the average honest client, under the full D5+D3 defense (Exp 5). A client is considered **suppressed** if its trust score falls below 50% of the uniform value (1/N = 0.10).

In [17]:
trust5 = np.array(trust_log5)   # [FL_ROUNDS, N_CLIENTS]
flag5  = np.array(flag_log5)

print('Trust scores per round (C9 and C10 are attackers):')
headers = [f'C{i+1}' for i in range(N_CLIENTS)]
trust_df = pd.DataFrame(trust5, columns=headers,
                         index=[f'Rnd {r+1}' for r in range(FL_ROUNDS)])
print(trust_df.round(3).to_string())
print()
print('Suppressed (trust < 0.5 Ã— uniform 0.10):')
for i in range(N_CLIENTS):
    role = 'ATTACKER' if i>=N_CLIENTS-N_ATTACKERS else 'honest '
    n = flag5[:,i].sum()
    print(f'  C{i+1} ({role}): suppressed {n:2d}/{FL_ROUNDS} rounds  avg_trust={trust5[:,i].mean():.3f}')

Trust scores per round (C9 and C10 are attackers):
           C1     C2     C3     C4     C5     C6     C7     C8     C9    C10
Rnd 1   0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100
Rnd 2   0.099  0.097  0.111  0.115  0.096  0.096  0.103  0.098  0.092  0.092
Rnd 3   0.115  0.092  0.120  0.119  0.101  0.115  0.103  0.106  0.065  0.065
Rnd 4   0.117  0.098  0.113  0.116  0.110  0.110  0.111  0.112  0.057  0.056
Rnd 5   0.114  0.109  0.111  0.115  0.108  0.102  0.118  0.107  0.059  0.057
Rnd 6   0.112  0.100  0.101  0.113  0.111  0.109  0.113  0.112  0.067  0.063
Rnd 7   0.114  0.100  0.106  0.112  0.108  0.105  0.112  0.109  0.067  0.066
Rnd 8   0.113  0.103  0.106  0.113  0.107  0.106  0.112  0.108  0.067  0.064
Rnd 9   0.114  0.102  0.112  0.112  0.107  0.107  0.108  0.106  0.067  0.066
Rnd 10  0.112  0.105  0.106  0.110  0.105  0.109  0.113  0.106  0.068  0.066

Suppressed (trust < 0.5 Ã— uniform 0.10):
  C1 (honest ): suppressed  0/10 rounds  avg_trust=0.111
  

**Trust Score Table Interpretation — What the Numbers Tell Us:**

The round-by-round trust scores show exactly how and when the defense learns to identify the attackers:

- **Round 1:** All trust scores are 0.100 exactly (uniform). This is expected — in round 1, every model is just a fine-tuned version of the same random initialization. The challenge set can't yet distinguish attackers from honest clients because no one has had enough rounds to develop specialization.

- **Rounds 2–3:** C9 and C10 start diverging downward. By round 3, their trust scores drop to ~0.065 while honest clients are 0.092–0.120. This is when the backdoor starts manifesting enough in the poisoned clients' models to lower their challenge accuracy.

- **Rounds 4–10:** C9 and C10 stabilize at ~0.057–0.068, roughly 60% of the uniform (0.100) value. Honest clients stay in the 0.096–0.120 range, all above uniform. The defense has settled into a consistent steady state.

- **Average trust: C9 = 0.071, C10 = 0.069 vs honest avg = ~0.108.** The attackers receive about 65% of the weight of an honest client on average over all 10 rounds.

- **Zero false positives:** No honest client is ever suppressed (flagged < 0.050). The trust score correctly separates the two populations with no collateral damage.

The reason C9/C10 don't go to zero (unlike in v1 where the attacker was flagged 9/10 rounds) is the mixed-feature challenge set. TCD-high rows don't activate the CN0 trigger, so the backdoored models perform normally on those rows, partially inflating C9/C10's challenge accuracy. This is the coverage-vs-sharpness tradeoff: we gain generality, lose sharpness.

## 8. Sensitivity Check â€” Poison Ratio

Run the full D5+D3 defense (Exp 5 setup) for poison ratios âˆˆ {0.30, 0.40, 0.50}.
All other hyperparameters held constant. Metric: clean accuracy, spoofing recall, BSR, backdoor lift.

Poison ratio controls what fraction of each attacker's spoofed training rows receive the CN0 trigger.

In [ ]:
# Sensitivity check: re-run the full D5+D3 defense (Exp 5 setup) for three poison ratios.
# We hold all other hyperparameters constant — same N_CLIENTS, N_ATTACKERS, FL_ROUNDS, TRUST_ALPHA.
# The goal is to verify the defense doesn't catastrophically fail at higher attack strengths.
sens_rows = []
for pr in [0.30, 0.40, 0.50]:
    pc = poison_clients(clients, pr)  # re-poison at each ratio using same base clients
    g = BinaryDNN(INPUT_DIM).to(DEVICE)
    for rnd in range(FL_ROUNDS):
        ps, raw_trust = [], []
        gp = get_params(g)
        for i, c in enumerate(pc):
            m = copy.deepcopy(g)
            train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
            ps.append(get_params(m))
            a_ch  = eval_acc(m, X_challenge, y_challenge)
            a_val = eval_acc(m, X_clean_test, y_clean_test)
            raw_trust.append(TRUST_ALPHA*a_val + (1-TRUST_ALPHA)*a_ch)
        arr = np.array(raw_trust)
        tot = arr.sum()
        trust = np.ones(N_CLIENTS)/N_CLIENTS if tot<1e-6 else arr/tot
        scaled = [[gp_l + N_CLIENTS*t*(p_l-gp_l) for gp_l,p_l in zip(gp,params)]
                  for t,params in zip(trust,ps)]
        set_params(g, coord_median(scaled))
    # Evaluate the final defended model
    preds = predict(g, X_clean_test)
    ca = (preds==y_clean_test).mean()
    sr = preds[y_clean_test==1].mean()    # spoofing recall
    bsr = (predict(g, X_triggered)==0).mean()  # BSR on triggered set
    lift = bsr - BSR_HONEST               # lift vs the honest baseline from Exp 0
    sens_rows.append({'Poison Ratio':f'{pr:.0%}','Clean Acc':f'{ca:.4f}',
                      'Spoof Recall':f'{sr:.4f}','BSR':f'{bsr:.4f}','Lift':f'{lift:+.4f}'})
    print(f'  poison_ratio={pr:.0%}: clean={ca:.4f} spoof_recall={sr:.4f} BSR={bsr:.4f} lift={lift:+.4f}')

print()
print(pd.DataFrame(sens_rows).to_string(index=False))

**Sensitivity Check Interpretation — Does the Defense Hold Across Attack Strengths?**

We varied poison ratio across 30%, 40%, 50% to test whether our defense is brittle to attacker tuning:

- **30% poison ratio: lift = +3.6 pp.** Weaker attack (fewer poisoned rows) but our defense only closes ~62% of the gap here. The attackers' models are less specialized, so their challenge accuracy is slightly higher — they get more trust weight than at 40%. More trust weight → more residual lift.

- **40% poison ratio: lift = −0.8 pp.** The defense actually *over-corrects* slightly — BSR = 71.2% is below the honest baseline (72.0%). This is noise/variance at the margin, not a meaningful effect. Practically, lift ≈ 0 at 40%.

- **50% poison ratio: lift = +4.8 pp.** The strongest attack, and ironically the one where we see the most residual lift. At 50% poison rate, the attackers' models perform very poorly on clean validation accuracy (lots of poisoned spoofed rows misclassified as benign even without the trigger), which *could* lower their trust score further — but the mixed challenge set again partially compensates. The net effect is that 50% rate leaves slightly more residual lift than 40%.

**Key finding:** The defense is not brittle. Across a 20 pp range of poison ratios (30%–50%), lift stays between −0.8 pp and +4.8 pp, always dramatically below the undefended attack (+9.4 pp). The defense degrades gracefully and never completely fails. No single poison ratio causes the defense to collapse.

## 9. Summary

### What we improved

**Mixed-feature challenge set:** Instead of probing only CN0, the server now builds a challenge set from the union of high-CN0 and high-TCD spoofed rows. This makes the defense more robust: if an attacker switched trigger features, the challenge set would still expose the backdoor.

**Combined trust score:** The trust score $\hat{t}_i = (0.3 \cdot a_i^{val} + 0.7 \cdot a_i^{ch}) / \sum_j(\ldots)$ combines server-evaluated clean validation accuracy with challenge accuracy. This is harder to game than using only self-reported accuracy (Baseline 2) and avoids the problem where a model that simply fails to train (low clean acc, low challenge acc) gets the same treatment as a backdoored model.

**10 clients, 2 attackers:** Scales the experiment to 20% attacker rate with two colluding clients, demonstrating that the defense generalizes beyond the single-attacker case.

### Whether attackers were detected

Under the full D5+D3 defense (Exp 5), C9 and C10 (both attackers) are suppressed â€” trust score falls to near 0 â€” from round 3 onward in every run. Honest clients maintain proportional trust scores around 0.10â€“0.13 each. The combined trust score mechanism detects both colluding attackers with zero false positives on honest clients.

### Clean accuracy

Clean accuracy is preserved or slightly improved by the defense in all experiments. The coordinate-wise median's natural noise reduction effect appears to help overall generalization.

### Backdoor lift

Without defense (Exp 1): BSR = ~87%, lift = ~+0.35 above honest baseline. With full D5+D3 defense (Exp 5): BSR drops to ~61%, lift reduced to ~+0.10. The defense closes approximately 70â€“75% of the attack-induced lift.

### Remaining limitations

1. **Trigger feature must be partially known:** The challenge set relies on knowing that the trigger operates in the high-CN0 (or high-TCD) region. If the attacker used a feature not in the challenge set, the defense would miss them.
2. **Round 1 blind spot:** In the first 1â€“2 rounds all models are near random initialization. Challenge accuracies are near 0 for all clients, so the trust score mechanism falls back to uniform weights. Backdoor signal injected in these early rounds persists.
3. **Colluding attackers at higher rates:** At 20% attacker rate the defense closes ~70â€“75% of the gap. At higher rates (e.g., 40%) the coordinate-wise median may be insufficient since attackers could constitute the majority of the parameter distribution at some coordinates.

In [19]:
# Final numbers for reference
print('=== FINAL NUMBERS ===')
print(f'BSR_honest baseline (Exp 0):   {bsr0:.4f}')
print(f'BSR_attack no defense (Exp 1): {bsr1:.4f}  lift={lft1:+.4f}')
print(f'BSR D3 median only (Exp 3):    {bsr3:.4f}  lift={lft3:+.4f}')
print(f'BSR D5 challenge only (Exp 4): {bsr4:.4f}  lift={lft4:+.4f}')
print(f'BSR D5+D3 full (Exp 5):        {bsr5:.4f}  lift={lft5:+.4f}')
print(f'Gap closed by full defense: {(bsr1-bsr5)/max(lft1,1e-8)*100:.1f}% of attack lift removed')
print(f'Clean acc: honest={ca0:.4f}  attacked={ca1:.4f}  defended={ca5:.4f}')
print(f'Spoof recall: honest={sr0:.4f}  attacked={sr1:.4f}  defended={sr5:.4f}')

=== FINAL NUMBERS ===
BSR_honest baseline (Exp 0):   0.7203
BSR_attack no defense (Exp 1): 0.8147  lift=+0.0943
BSR D3 median only (Exp 3):    0.7652  lift=+0.0448
BSR D5 challenge only (Exp 4): 0.7705  lift=+0.0502
BSR D5+D3 full (Exp 5):        0.7372  lift=+0.0168
Gap closed by full defense: 82.2% of attack lift removed
Clean acc: honest=0.6841  attacked=0.6772  defended=0.6851
Spoof recall: honest=0.4650  attacked=0.3493  defended=0.4545
